# Difix Before/After — nur aus den in Drive liegenden Generierungen

Schlankes Standalone-Notebook fuer den praktischen Difix-Teil. Es macht **kein**
Frame-Extrahieren, **kein** COLMAP-Matching und **kein** Training, sondern laedt
die bereits im Drive liegenden Artefakte (Checkpoint + COLMAP-Cache), rendert
Novel Views und saeubert sie mit Difix.

## Voraussetzungen
1. **FRISCHER Runtime.** Vor dem Start: *Laufzeit -> Trennen und loeschen*
   (NICHT nur "Sitzung neu starten"!). Grund: ein in einer frueheren Session
   ueberschriebenes `torch` ueberlebt einen Kernel-Neustart und zerlegt dann
   gsplat (`gsplat_cuda.so: cannot open shared object file`) und Difix
   (`torch._dynamo ... has no attribute decorators`). Nur eine neue VM hat
   wieder Colabs Original-torch. Schritt 2 prueft das und bricht sonst mit
   klarer Meldung ab.
2. Die Szene wurde im Hauptnotebook (`room_to_3d.ipynb`) **schon trainiert** —
   `…/3dgs_output/gsplat_base/ckpts/ckpt_*.pt` und `…/3dgs_output/colmap_cache/…`
   liegen in Drive.

Dieses Notebook fasst Colabs torch/CUDA-Stack bewusst NICHT an (kein
`requirements.txt`, kein xformers) — genau das war die Ursache der bisherigen
Schritt-9-Fehler.

In [ ]:
# === Schritt 1: Drive einbinden, Szene + Artefakte finden ===
from google.colab import drive
drive.mount('/content/drive')

import os, glob, torch

SEMINAR_ROOT  = '/content/drive/MyDrive/Seminar'
SCENE_SELECT  = 1            # gleiche Szenen-Nummer wie im Hauptnotebook
OUTPUT_SUBDIR = '3dgs_output'
DATA_FACTOR   = 4            # muss zum Training passen
SCENE_DIR     = '/content/scene'

# Szenen-Ordner aufloesen (Zahl -> Ordnername, str -> Teilstring) wie im Hauptnotebook
assert os.path.isdir(SEMINAR_ROOT), f'SEMINAR_ROOT fehlt: {SEMINAR_ROOT} (Drive gemountet?)'
_scenes = sorted(d for d in os.listdir(SEMINAR_ROOT)
                 if os.path.isdir(os.path.join(SEMINAR_ROOT, d)) and d != OUTPUT_SUBDIR)
if isinstance(SCENE_SELECT, int):
    _n = str(SCENE_SELECT)
    _cand = [d for d in _scenes if d == _n or any(d.startswith(_n + s) for s in ('.', ' ', '-', '_'))]
else:
    _cand = [d for d in _scenes if SCENE_SELECT.lower() in d.lower()]
assert len(_cand) == 1, f'SCENE_SELECT={SCENE_SELECT!r}: Treffer={_cand}, vorhanden={_scenes}'
SCENE_NAME   = _cand[0]
DRIVE_OUTPUT = os.path.join(SEMINAR_ROOT, SCENE_NAME, OUTPUT_SUBDIR)
RESULT_DIR   = f'{DRIVE_OUTPUT}/gsplat_base'

# Checkpoint (letzter ckpt_*.pt)
_ckpts = sorted(glob.glob(f'{RESULT_DIR}/ckpts/ckpt_*.pt'))
assert _ckpts, (f'Kein Checkpoint in {RESULT_DIR}/ckpts/ - erst im Hauptnotebook '
                f'Schritt 7 fuer Szene "{SCENE_NAME}" trainieren.')
CKPT = _ckpts[-1]

# COLMAP-Cache (Posen+Bilder-Paar, vor Undistortion gesichert)
_cc = sorted(glob.glob(f'{DRIVE_OUTPUT}/colmap_cache/*/sparse/0/cameras.bin'))
assert _cc, (f'Kein COLMAP-Cache in {DRIVE_OUTPUT}/colmap_cache/ - im Hauptnotebook '
             'muss Schritt 5 (COLMAP) einmal gelaufen + gecacht sein.')
COLMAP_CACHE = os.path.dirname(os.path.dirname(os.path.dirname(_cc[-1])))  # .../colmap_cache/{hash}

DRIVE_DIFIX = f'{DRIVE_OUTPUT}/difix_standalone'
os.makedirs(DRIVE_DIFIX, exist_ok=True)

print('Szene         :', SCENE_NAME)
print('Checkpoint    :', CKPT)
print('COLMAP-Cache  :', COLMAP_CACHE)
print('Output -> Drive:', DRIVE_DIFIX)
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU:', torch.cuda.is_available())

In [ ]:
# === Schritt 2: Dependencies - Colabs torch/CUDA-Stack NICHT anfassen ===
import torch

# torch-GESUNDHEITS-CHECK ZUERST. Haeufigster Stolperstein: ein in einer vorigen
# Session ueberschriebener torch ueberlebt "Sitzung neu starten". Wenn das hier
# (oder Schritt 3/5/6) crasht -> Laufzeit "Trennen und loeschen" und neu starten.
assert torch.cuda.is_available(), (
    'GPU/torch nicht verfuegbar. Wenn torch frueher ueberschrieben wurde: Laufzeit '
    '-> "Trennen und loeschen" (NICHT nur "Sitzung neu starten") und neu ausfuehren.')

# System: COLMAP fuer die Undistortion der gecachten Posen.
!apt-get -qq install -y colmap > /dev/null && echo 'COLMAP installiert.'

# gsplat - CUDA-Kernel werden beim ersten rasterization()-Aufruf einmalig JIT-
# kompiliert (~1-3 Min, auf frischem torch problemlos). KEINE numpy-Pins.
!pip install -q gsplat==1.5.3
!pip install -q viser nerfview tyro 'imageio[ffmpeg]' tqdm Pillow opencv-python plyfile splines ninja

# Difix: NUR die vier Python-Pins (alte diffusers-API). KEIN torch/torchvision/
# xformers -> torch bleibt unangetastet. hf-hub muss <0.26 bleiben (diffusers
# 0.25.1 nutzt cached_download). ZULETZT, damit hf-hub gepinnt bleibt.
!pip install -q diffusers==0.25.1 transformers==4.38.0 huggingface-hub==0.25.1 peft==0.9.0

import numpy as _np
assert _np.__version__.startswith('2.'), f'numpy unerwartet {_np.__version__} (sollte 2.x sein).'
import torch._dynamo as _dyn  # noqa: F401  -> wirft frueh, falls torch inkonsistent
print('OK | torch', torch.__version__, '| numpy', _np.__version__, '| CUDA ok')

In [ ]:
# === Schritt 3: gsplat-Beispiel-Dataloader + pycolmap (numpy-2-Patch) ===
# datasets.colmap.Parser noetig, weil der Checkpoint mit dessen normalize=True-
# Szenen-Normalisierung trainiert wurde -> nur so rendern wir im gleichen Raum.
%cd /content
GSPLAT_VERSION = 'v1.5.3'
import shutil
if os.path.exists('/content/gsplat'):
    _head = ''
    try: _head = open('/content/gsplat/.git/HEAD').read().strip()
    except Exception: pass
    if GSPLAT_VERSION not in _head:
        shutil.rmtree('/content/gsplat')
if not os.path.exists('/content/gsplat'):
    !git clone --depth 1 --branch {GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git

# examples/datasets/ braucht __init__.py, sonst gewinnt HuggingFace-'datasets'.
_init = '/content/gsplat/examples/datasets/__init__.py'
if not os.path.exists(_init):
    open(_init, 'w').close()

# pycolmap-Fork (Pure-Python COLMAP-Reader; anderes Paket als PyPI-pycolmap).
!pip install -q --no-deps 'pycolmap @ git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e' || echo 'WARNUNG: pycolmap-Install fehlgeschlagen.'

# PATCH: scene_manager.py macht np.uint64(-1) -> OverflowError unter numpy 2.x.
# find_spec lokalisiert die Datei OHNE sie zu importieren (Import wuerde crashen).
import importlib.util as _ilu
_sm = None
_spec = _ilu.find_spec('pycolmap')
if _spec and _spec.origin:
    _c = os.path.join(os.path.dirname(_spec.origin), 'scene_manager.py')
    if os.path.exists(_c): _sm = _c
if _sm is None:
    _h = glob.glob('/usr/local/lib/python3.*/dist-packages/pycolmap/scene_manager.py')
    _sm = _h[0] if _h else None
if _sm:
    _src = open(_sm).read()
    if 'np.uint64(-1)' in _src:
        open(_sm, 'w').write(_src.replace('np.uint64(-1)', 'np.uint64(2**64 - 1)'))
        print('[OK] pycolmap/scene_manager.py gegen numpy-2 gepatcht.')
    else:
        print('[SKIP] pycolmap bereits numpy-2-tauglich.')
else:
    print('[WARN] pycolmap/scene_manager.py nicht gefunden.')
print('gsplat-Dataloader + pycolmap bereit.')

In [ ]:
# === Schritt 4: /content/scene aus dem Drive-COLMAP-Cache aufbauen ===
# Cache haelt die DISTORTED Posen+Bilder; wir warpen sie - exakt wie Schritt 5b
# im Hauptnotebook - auf PINHOLE, weil der Checkpoint darauf trainiert wurde.
import shutil, struct
from PIL import Image
from tqdm import tqdm

shutil.rmtree(SCENE_DIR, ignore_errors=True)
os.makedirs(SCENE_DIR)
shutil.copytree(f'{COLMAP_CACHE}/sparse', f'{SCENE_DIR}/sparse')
shutil.copytree(f'{COLMAP_CACHE}/images', f'{SCENE_DIR}/images')
sparse_dir = f'{SCENE_DIR}/sparse/0'

def _first_camera_model_id(bin_path):
    with open(bin_path, 'rb') as _f:
        _f.read(8); _f.read(4)                  # num_cameras (uint64), camera_id (uint32)
        return struct.unpack('<i', _f.read(4))[0]

cam_model = _first_camera_model_id(f'{sparse_dir}/cameras.bin')
if cam_model in (0, 1):
    print(f'[SKIP] Camera-Modell bereits PINHOLE (id={cam_model}).')
else:
    print(f'[Undistort] Camera-Modell id={cam_model} -> PINHOLE...')
    undist_tmp = f'{SCENE_DIR}/_undist_tmp'
    shutil.rmtree(undist_tmp, ignore_errors=True)
    !colmap image_undistorter --image_path {SCENE_DIR}/images --input_path {sparse_dir} --output_path {undist_tmp} --output_type COLMAP --max_image_size 2000
    if not os.path.exists(f'{undist_tmp}/sparse/cameras.bin'):
        raise RuntimeError('colmap image_undistorter ist gescheitert.')
    shutil.rmtree(f'{SCENE_DIR}/images')
    shutil.move(f'{undist_tmp}/images', f'{SCENE_DIR}/images')
    shutil.rmtree(f'{SCENE_DIR}/sparse')
    os.makedirs(f'{SCENE_DIR}/sparse/0', exist_ok=True)
    for fn in os.listdir(f'{undist_tmp}/sparse'):
        shutil.move(f'{undist_tmp}/sparse/{fn}', f'{SCENE_DIR}/sparse/0/{fn}')
    shutil.rmtree(undist_tmp, ignore_errors=True)
    print(f'[OK] Undistortion fertig (Modell-id jetzt {_first_camera_model_id(f"{sparse_dir}/cameras.bin")}).')

# Downscale images_{DATA_FACTOR} (gsplat erwartet die heruntergerechnete Variante)
_srcd = f'{SCENE_DIR}/images'
_n = len(os.listdir(_srcd))
_dstd = f'{SCENE_DIR}/images_{DATA_FACTOR}'
if os.path.isdir(_dstd) and len(os.listdir(_dstd)) >= _n:
    print(f'[SKIP] images_{DATA_FACTOR} existiert ({len(os.listdir(_dstd))}).')
else:
    os.makedirs(_dstd, exist_ok=True)
    for fn in tqdm(sorted(os.listdir(_srcd)), desc=f'images_{DATA_FACTOR}'):
        im = Image.open(os.path.join(_srcd, fn)); w, h = im.size
        im.resize((max(1, w // DATA_FACTOR), max(1, h // DATA_FACTOR)), Image.LANCZOS).save(
            os.path.join(_dstd, fn), quality=92)
print('Szene bereit ->', sorted(os.listdir(SCENE_DIR)))

## Schritt 5: Difix Mini-Demo (Beweis, dass das Modell laeuft)

Saeubert ein mitgeliefertes Beispiel-Render aus dem Difix3D-Repo. Braucht
**weder gsplat noch den Checkpoint** (reiner diffusers-Pfad) und laedt das
~5.2-GB-Modell `nvidia/difix`.

In [ ]:
# === Schritt 5: Difix Mini-Demo ===
%cd /content
import os
if not os.path.exists('/content/Difix3D'):
    !git clone --depth 1 https://github.com/nv-tlabs/Difix3D.git

import sys
from PIL import Image
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/Difix3D/src')
from pipeline_difix import DifixPipeline

difix = DifixPipeline.from_pretrained('nvidia/difix', trust_remote_code=True)
difix.to('cuda')

_inp = Image.open('/content/Difix3D/assets/example_input.png').convert('RGB')
_out = difix('remove degradation', image=_inp, num_inference_steps=1,
             timesteps=[199], guidance_scale=0.0).images[0]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(_inp); ax[0].set_title('Vorher (Beispiel-Render mit Artefakten)')
ax[1].imshow(_out); ax[1].set_title('Nachher: Difix (1 Schritt)')
for a in ax: a.axis('off')
plt.show()
print('Difix laeuft.')

## Schritt 6: Before/After auf der EIGENEN Szene

Rendert eine Novel-View-Trajektorie aus dem Checkpoint (Before) und laesst Difix
jedes Frame saeubern (After). Ergebnis: `before_after_difix.mp4` + Einzelbild-
Paare, beides nach Drive (`…/3dgs_output/difix_standalone/`).

**Hinweis:** Beim ersten `rasterization()`-Aufruf kompiliert gsplat seine
CUDA-Kernel einmalig (~1-3 Min, Spinner). Das ist normal, nicht haengen
geblieben — auf einem frischen torch laeuft es durch.

In [ ]:
# === Schritt 6: Before/After auf der eigenen Szene ===
import os, sys, glob, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from scipy.spatial.transform import Rotation, Slerp

# Difix3D-Repo + difix-Pipeline sicherstellen (falls Schritt 5 uebersprungen wurde)
if not os.path.exists('/content/Difix3D'):
    !git clone --depth 1 https://github.com/nv-tlabs/Difix3D.git
sys.path.insert(0, '/content/Difix3D/src')

sys.modules.pop('datasets', None)  # falls HF-datasets schon (an)importiert wurde
sys.path.insert(0, '/content/gsplat/examples')
from datasets.colmap import Parser, Dataset
from gsplat.rendering import rasterization

NUM_NOVEL   = 60     # Frames (T4: ~3-6 Min Difix-Zeit)
LATERAL_OFF = 0.25   # seitlicher Versatz in Einheiten der Szenen-Skala
device = 'cuda'

# trusted eigener Checkpoint -> weights_only=False (gsplat-ckpt ist kein reines state_dict)
splats = torch.load(CKPT, map_location=device, weights_only=False)['splats']
means  = splats['means'].float()
quats  = splats['quats'].float()
scales = torch.exp(splats['scales'].float())
opac   = torch.sigmoid(splats['opacities'].float())
shs    = torch.cat([splats['sh0'], splats['shN']], 1).float()

# Gleiche Parser-Einstellungen wie im Training (normalize=True)
p = Parser(SCENE_DIR, factor=DATA_FACTOR, normalize=True, test_every=8)
d0 = Dataset(p, split='train')[0]
K = d0['K'].float()[None].to(device)
H, W = d0['image'].shape[:2]

# Novel-Trajektorie: Anker-Posen seitlich + leicht nach oben versetzen
c2ws = p.camtoworlds
scene_scale = float(p.scene_scale)
anchors = c2ws[:: max(1, len(c2ws) // 8)]
if len(anchors) < 2:
    anchors = c2ws[:2]
positions, rots, times = [], [], []
for i, m in enumerate(anchors):
    right = m[:3, 0]; up_w = -m[:3, 1]   # OpenCV: y zeigt nach unten
    positions.append(m[:3, 3] + LATERAL_OFF * scene_scale * right + 0.10 * scene_scale * up_w)
    rots.append(m[:3, :3]); times.append(i)
slerp = Slerp(times, Rotation.from_matrix(np.stack(rots)))
ts = np.linspace(0, times[-1], NUM_NOVEL)
pos_i = np.stack([np.interp(ts, times, np.array(positions)[:, k]) for k in range(3)], 1)
rot_i = slerp(ts).as_matrix()

RENDER_DIR = '/content/novel/render'; FIXED_DIR = '/content/novel/fixed'
for dd in (RENDER_DIR, FIXED_DIR):
    shutil.rmtree(dd, ignore_errors=True); os.makedirs(dd)

# Before: 3DGS rendert die Novel Views (hier kompiliert gsplat seine CUDA-Kernel einmalig)
with torch.no_grad():
    for i in range(NUM_NOVEL):
        c2w = np.eye(4); c2w[:3, :3] = rot_i[i]; c2w[:3, 3] = pos_i[i]
        viewmat = torch.linalg.inv(torch.tensor(c2w, dtype=torch.float32, device=device))[None]
        img, _, _ = rasterization(means, quats, scales, opac, shs, viewmat, K, W, H, sh_degree=3)
        arr = (img[0].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
        Image.fromarray(arr).resize((1024, 576), Image.LANCZOS).save(f'{RENDER_DIR}/{i:04d}.png')
print(f'[OK] {NUM_NOVEL} Novel Views gerendert.')

del splats, means, quats, scales, opac, shs
torch.cuda.empty_cache()

# difix wurde evtl. in Schritt 5 schon geladen
if 'difix' not in dir():
    from pipeline_difix import DifixPipeline
    difix = DifixPipeline.from_pretrained('nvidia/difix', trust_remote_code=True); difix.to(device)

# After: Difix ueber jedes Frame (1 Diffusionsschritt pro Bild)
for fp in tqdm(sorted(glob.glob(f'{RENDER_DIR}/*.png')), desc='Difix'):
    im = Image.open(fp).convert('RGB')
    out = difix('remove degradation', image=im, num_inference_steps=1,
                timesteps=[199], guidance_scale=0.0).images[0]
    out.save(f'{FIXED_DIR}/{os.path.basename(fp)}')

# Side-by-Side-Video + Beispiel-Frames nach Drive
BA_VIDEO = f'{DRIVE_DIFIX}/before_after_difix.mp4'
!ffmpeg -y -framerate 15 -i {RENDER_DIR}/%04d.png -framerate 15 -i {FIXED_DIR}/%04d.png -filter_complex "[0:v][1:v]hstack=inputs=2" -c:v libx264 -pix_fmt yuv420p {BA_VIDEO} -loglevel error

pairs = f'{DRIVE_DIFIX}/before_after_frames'; os.makedirs(pairs, exist_ok=True)
for i in np.linspace(0, NUM_NOVEL - 1, 3).astype(int):
    shutil.copy(f'{RENDER_DIR}/{i:04d}.png', f'{pairs}/{i:04d}_before.png')
    shutil.copy(f'{FIXED_DIR}/{i:04d}.png',  f'{pairs}/{i:04d}_after.png')

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].imshow(Image.open(f'{RENDER_DIR}/0000.png')); ax[0].set_title('Before: 3DGS Novel View')
ax[1].imshow(Image.open(f'{FIXED_DIR}/0000.png'));  ax[1].set_title('After: + Difix')
for a in ax: a.axis('off')
plt.show()
print('Before/After-Video:', BA_VIDEO)
print('Einzelbild-Paare  :', pairs)